In [1]:
import boto3
import os
from botocore.exceptions import NoCredentialsError, ClientError

In [ ]:
BUCKET_NAME="bucket-name"
LOCAL_DIRECTORY = './mis_archivos/'
REGION = "us-west-2"  

# Crear el directorio local si no existe para evitar errores
if not os.path.exists(LOCAL_DIRECTORY):
    os.makedirs(LOCAL_DIRECTORY)
    print(f"Directorio creado: {LOCAL_DIRECTORY}")

# Inicializar el cliente de S3
s3_client = boto3.client('s3')

In [10]:
# Función para crear bucket
def create_s3_bucket(bucket_name, region=None):
    try:
        if region is None:
            s3_client.create_bucket(Bucket=bucket_name)
        else:
            location = {'LocationConstraint': region}
            s3_client.create_bucket(Bucket=bucket_name,
                                    CreateBucketConfiguration=location)
        print(f"Bucket '{bucket_name}' creado exitosamente.")
        return True
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == 'BucketAlreadyOwnedByYou':
            print(f"El bucket '{bucket_name}' ya te pertenece.")
            return True
        elif error_code == 'BucketAlreadyExists':
            print(f"Error: El nombre '{bucket_name}' ya está en uso por otro usuario. Elige uno diferente.")
        else:
            print(f"Error al crear el bucket: {e}")
        return False

create_s3_bucket(BUCKET_NAME, REGION)

Bucket 's3-sync-project-vv' creado exitosamente.


True

In [11]:
# Función para listar objetos
def list_s3_files(bucket):
    """Lista los objetos presentes en el bucket de S3."""
    try:
        response = s3_client.list_objects_v2(Bucket=bucket)
        if 'Contents' in response:
            return [obj['Key'] for obj in response['Contents']]
        return []
    except ClientError as e:
        print(f"Error al listar archivos: {e}")
        return []

# Ejemplo de uso:
archivos_en_s3 = list_s3_files(BUCKET_NAME)
print(f"Archivos en S3: {archivos_en_s3}")

Archivos en S3: []


In [13]:
# Función de subida de archivos
def upload_missing_files(directory, bucket):
    """Sube archivos locales a S3 si no están presentes en el bucket."""
    s3_files = list_s3_files(bucket)
    local_files = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]
    
    for file in local_files:
        if file not in s3_files:
            try:
                print(f"Subiendo {file}...")
                s3_client.upload_file(os.path.join(directory, file), bucket, file)
                print(f" {file} subido exitosamente.")
            except Exception as e:
                print(f"Error al subir {file}: {e}")
        else:
            print(f" {file} ya existe en S3. Omitiendo.")

# Ejecutar subida
upload_missing_files(LOCAL_DIRECTORY, BUCKET_NAME)

#Listamos los archivos 
archivos_en_s3 = list_s3_files(BUCKET_NAME)
print(f"Archivos en S3: {archivos_en_s3}")

Subiendo data.csv...
 data.csv subido exitosamente.
Archivos en S3: ['data.csv', 'data_csv']


In [ ]:
# Función descarga de archivos
def download_missing_files(directory, bucket):
    """Descarga archivos de S3 si no están presentes en el directorio local."""
    s3_files = list_s3_files(bucket)
    local_files = os.listdir(directory)
    
    for s3_file in s3_files:
        if s3_file not in local_files:
            try:
                print(f"Descargando {s3_file}...")
                s3_client.download_file(bucket, s3_file, os.path.join(directory, s3_file))
                print(f"{s3_file} descargado exitosamente.")
            except Exception as e:
                print(f"Error al descargar {s3_file}: {e}")
        else:
            print(f"{s3_file} ya existe localmente. Omitiendo.")

# Ejecutar descarga
download_missing_files(LOCAL_DIRECTORY, BUCKET_NAME)

ℹ️ data.csv ya existe localmente. Omitiendo.
Descargando data_csv...
✅ data_csv descargado exitosamente.


In [ ]:
# Función main
def main():
    print("--- Iniciando Automatizador S3 ---")
    
    # 1. Asegurar que el bucket exista
    if create_s3_bucket(BUCKET_NAME, REGION):
        
        print("\n2. Verificando subidas pendientes...")
        upload_missing_files(LOCAL_DIRECTORY, BUCKET_NAME)
        
        print("\n3. Verificando descargas pendientes...")
        download_missing_files(LOCAL_DIRECTORY, BUCKET_NAME)
        
        print("\n4. Estado final del Bucket:")
        print(list_s3_files(BUCKET_NAME))
    else:
        print("\n No se pudo proceder debido a un error con el Bucket.")
    
    print("\n--- Proceso Finalizado ---")

if __name__ == "__main__":
    main()

--- Iniciando Automatizador S3 ---
ℹ️ El bucket 's3-sync-project-vv' ya te pertenece.

2. Verificando subidas pendientes...
 data.csv ya existe en S3. Omitiendo.

3. Verificando descargas pendientes...
ℹ️ data.csv ya existe localmente. Omitiendo.
Descargando data_csv...
✅ data_csv descargado exitosamente.

4. Estado final del Bucket:
['data.csv', 'data_csv']

--- Proceso Finalizado ---
